# AI Impact on Job Market - Simplified Task2 Visualization

## Python Implementation (Plotly) - Simplified Version of React/D3 Task2

This notebook provides a **simplified** Python implementation of the React-based Task2 visualization:

### ✅ Core Features Implemented:
- **Bubble Chart**: AI Intensity vs Salary with industry comparison
- **Interactive Year Slider**: Replace D3 drag handles with Plotly widgets.Slider
- **Parallel Coordinates**: Multi-metric time series trends
- **Industry Filtering**: Click legend to show/hide industries
- **Trace Visualization**: Independent chart showing complete evolution paths

### ❌ Features Removed (Simplification):
- Tutorial/Guide system (Spotlight)
- Complex D3 transition animations
- Real-time trace rendering during slider drag
- Drag-to-select year range (replaced with single-year slider)

### Dependencies Required:
```bash
pip install pandas plotly numpy
```

In [14]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 1. Data Loading and Preprocessing

Load and aggregate raw job posting data by year and industry.

In [15]:
# Load CSV data
df_raw = None
try:
    df_raw = pd.read_csv('public/ai_impact_jobs_2010_2025.csv')
    print(f"✅ Data loaded successfully! Shape: {df_raw.shape}")
    print(f"\nColumns: {list(df_raw.columns)}")
except FileNotFoundError:
    print("❌ CSV file not found at 'public/ai_impact_jobs_2010_2025.csv'")
    print("\n⚠️ Generating sample data for demonstration...")
    
    # Generate realistic sample data
    np.random.seed(42)
    industries = ['Tech', 'Finance', 'Healthcare', 'Education', 'Manufacturing', 'Retail']
    years = list(range(2010, 2025))
    
    data = []
    for year in years:
        for industry in industries:
            n_jobs = np.random.randint(50, 200)
            # Simulate realistic trends
            base_salary = {
                'Tech': 85000, 'Finance': 75000, 'Healthcare': 70000,
                'Education': 55000, 'Manufacturing': 60000, 'Retail': 50000
            }
            salary_trend = base_salary[industry] * (1 + 0.03 * (year - 2010))
            ai_trend = 0.3 + 0.04 * (year - 2010) + np.random.uniform(-0.1, 0.1)
            
            data.extend([{
                'posting_year': year,
                'industry': industry,
                'salary_usd': np.random.normal(salary_trend, salary_trend * 0.15),
                'ai_intensity_score': np.clip(ai_trend, 0.1, 0.95),
                'automation_risk_score': np.random.uniform(0.15, 0.75),
            } for _ in range(n_jobs)])
    
    df_raw = pd.DataFrame(data)
    print(f"✅ Sample data generated! Shape: {df_raw.shape}")

display(df_raw.head())

✅ Data loaded successfully! Shape: (5000, 22)

Columns: ['job_id', 'posting_year', 'country', 'region', 'city', 'company_name', 'company_size', 'industry', 'job_title', 'seniority_level', 'ai_mentioned', 'ai_keywords', 'ai_intensity_score', 'core_skills', 'ai_skills', 'salary_usd', 'salary_change_vs_prev_year_percent', 'automation_risk_score', 'reskilling_required', 'ai_job_displacement_risk', 'job_description_embedding_cluster', 'industry_ai_adoption_stage']


,job_id,posting_year,country,region,city,company_name,company_size,industry,job_title,seniority_level,...,ai_intensity_score,core_skills,ai_skills,salary_usd,salary_change_vs_prev_year_percent,automation_risk_score,reskilling_required,ai_job_displacement_risk,job_description_embedding_cluster,industry_ai_adoption_stage
0,836b4774-702e-49ef-93d3-2f255ce1e910,2018,Brazil,South America,London,NextGen Technologies,Small,Education,Policy Analyst,Lead,...,0.81,"Research, Project Management, Business Analysis",reinforcement learning,61586,12.68,0.11,True,Low,14,Growing
1,43699e93-7b15-4728-a4c6-9e41ff438a25,2015,UAE,Middle East,Singapore,Future Solutions,Medium,Energy,Data Scientist,Executive,...,0.04,"Research, SQL, Business Analysis, Python, Clou...",NaN,62045,-3.98,0.71,False,High,19,Emerging
2,fc9d1854-3cbf-4bab-90df-77304dfc59df,2016,Nepal,South Asia,Sydney,Future Analytics,Startup,Finance,Product Manager,Junior,...,0.15,"Statistics, Project Management, Cloud Computin...",NaN,27035,3.55,0.86,False,High,2,Emerging
3,05c1c7d3-2add-4919-91eb-f6c78bfe23d1,2015,Spain,Europe,Nairobi,Global Technologies,Large,Government,Data Scientist,Mid,...,0.19,"Cloud Computing, SQL, Project Management, Comm...",NaN,72894,-2.80,0.70,False,Low,15,Emerging
4,5e739937-d1b0-44d7-935c-7ebb3fc1f6e8,2014,Taiwan,East Asia,Sydney,Future Technologies,Small,Manufacturing,ML Engineer,Lead,...,0.11,"SQL, Python, Communication, Software Engineeri...",NaN,57215,0.85,0.87,False,High,13,Emerging


In [16]:
# Aggregate data by year and industry (matching React's d3.rollups logic)
df_agg = df_raw.groupby(['posting_year', 'industry']).agg(
    avgSalary=('salary_usd', 'mean'),
    avgAIIntensity=('ai_intensity_score', 'mean'),
    avgAutomationRisk=('automation_risk_score', 'mean'),
    jobCount=('industry', 'size')
).reset_index()

# Rename columns to match React implementation
df_agg = df_agg.rename(columns={'posting_year': 'year'})

print(f"✅ Aggregated data shape: {df_agg.shape}")
print(f"📅 Year range: {df_agg['year'].min()} - {df_agg['year'].max()}")
print(f"🏭 Industries: {list(df_agg['industry'].unique())}")
print(f"\nSample aggregated data:")
display(df_agg.head(10))

✅ Aggregated data shape: (144, 6)
📅 Year range: 2010 - 2025
🏭 Industries: ['Agriculture', 'Education', 'Energy', 'Finance', 'Government', 'Healthcare', 'Manufacturing', 'Retail', 'Tech']

Sample aggregated data:


,year,industry,avgSalary,avgAIIntensity,avgAutomationRisk,jobCount
0,2010,Agriculture,47219.756098,0.105122,0.732927,41
1,2010,Education,48564.666667,0.107667,0.750000,30
2,2010,Energy,51022.413793,0.154483,0.728966,29
3,2010,Finance,55057.081081,0.205405,0.638919,37
4,2010,Government,47817.085106,0.172979,0.690000,47
5,2010,Healthcare,43619.054054,0.136486,0.721351,37
6,2010,Manufacturing,42342.707317,0.136098,0.698049,41
7,2010,Retail,47180.192308,0.149231,0.717692,26
8,2010,Tech,46424.820513,0.242564,0.606410,39
9,2011,Agriculture,46531.906250,0.097187,0.748125,32


## 2. Interactive Bubble Chart with Year Slider

This chart replaces the React D3 drag-handle interaction with a **Plotly Slider widget**.

### Key Features:
- **X-axis**: AI Intensity Score
- **Y-axis**: Average Salary (USD)
- **Bubble size**: Job Count (scaled with sqrt)
- **Bubble opacity**: Automation Risk (40%-80%)
- **Solid circles**: Selected year data
- **Arrow lines**: Show evolution from previous year
- **Slider**: Interactively change the displayed year

### How it works:
Instead of D3's complex drag event handlers, we use Plotly's `frames` and `sliders` to pre-compute all year states. When the user drags the slider, Plotly instantly switches between pre-rendered frames.

In [17]:
# Industry colors matching React implementation exactly
INDUSTRY_COLORS = {
    'Tech': '#dc2626',
    'Finance': '#d97706',
    'Healthcare': '#db2777',
    'Education': '#0891b2',
    'Manufacturing': '#7c3aed',
    'Retail': '#ea580c',
    'Agriculture': '#16a34a',
    'Energy': '#0d9488',
    'Government': '#f59e0b',
}

print("✅ Industry color palette defined")
for industry, color in INDUSTRY_COLORS.items():
    print(f"   {industry}: {color}")

✅ Industry color palette defined
   Tech: #dc2626
   Finance: #d97706
   Healthcare: #db2777
   Education: #0891b2
   Manufacturing: #7c3aed
   Retail: #ea580c
   Agriculture: #16a34a
   Energy: #0d9488
   Government: #f59e0b


In [25]:
def create_interactive_bubble_chart(df, show_trace=True):
    """
    Create interactive bubble chart with Plotly slider.
    Fixed: All frames have consistent trace structure.
    """
    
    years = sorted(df['year'].unique())
    industries = sorted(df['industry'].unique())
    
    fig = go.Figure()
    
    # Add initial traces (first year)
    first_year_data = df[df['year'] == years[0]].copy()
    
    for industry in industries:
        color = INDUSTRY_COLORS.get(industry, '#64748b')
        
        if industry in first_year_data['industry'].values:
            row = first_year_data[first_year_data['industry'] == industry].iloc[0]
            size = float(np.sqrt(row['jobCount'])) * 2.5
            opacity = float(0.4 + row['avgAutomationRisk'] * 0.6)
            
            fig.add_trace(go.Scatter(
                x=[float(row['avgAIIntensity'])],
                y=[float(row['avgSalary'])],
                mode='markers',
                marker=dict(size=size, color=color, opacity=opacity, line=dict(color='white', width=2)),
                name=industry,
                text=f"{industry}<br>Year: {years[0]}<br>Salary: ${row['avgSalary']:.0f}<br>AI Intensity: {row['avgAIIntensity']:.2f}<br>Jobs: {int(row['jobCount'])}",
                hoverinfo='text',
                legendgroup=industry,
                showlegend=True
            ))
        else:
            fig.add_trace(go.Scatter(x=[], y=[], mode='markers', marker=dict(size=0), name=industry, legendgroup=industry, showlegend=True))
    
    # Create frames
    frames = []
    for year in years:
        year_data = df[df['year'] == year].copy()
        frame_list = []
        
        # Add bubble traces
        for industry in industries:
            color = INDUSTRY_COLORS.get(industry, '#64748b')
            
            if industry in year_data['industry'].values:
                row = year_data[year_data['industry'] == industry].iloc[0]
                size = float(np.sqrt(row['jobCount'])) * 2.5
                opacity = float(0.4 + row['avgAutomationRisk'] * 0.6)
                
                frame_list.append(go.Scatter(
                    x=[float(row['avgAIIntensity'])],
                    y=[float(row['avgSalary'])],
                    mode='markers',
                    marker=dict(size=size, color=color, opacity=opacity, line=dict(color='white', width=2)),
                    name=industry,
                    text=f"{industry}<br>Year: {year}<br>Salary: ${row['avgSalary']:.0f}<br>AI Intensity: {row['avgAIIntensity']:.2f}<br>Jobs: {int(row['jobCount'])}",
                    hoverinfo='text',
                    legendgroup=industry,
                    showlegend=False
                ))
            else:
                frame_list.append(go.Scatter(x=[], y=[], mode='markers', marker=dict(size=0), name=industry, legendgroup=industry, showlegend=False))
        
        # Add traces to beginning if not first year
        if show_trace and year > years[0]:
            prev_year_data = df[df['year'] == year - 1]
            trace_list = []
            
            for industry in industries:
                if industry in year_data['industry'].values and industry in prev_year_data['industry'].values:
                    curr_row = year_data[year_data['industry'] == industry].iloc[0]
                    prev_row = prev_year_data[prev_year_data['industry'] == industry].iloc[0]
                    color = INDUSTRY_COLORS.get(industry, '#64748b')
                    
                    trace_list.append(go.Scatter(
                        x=[float(prev_row['avgAIIntensity']), float(curr_row['avgAIIntensity'])],
                        y=[float(prev_row['avgSalary']), float(curr_row['avgSalary'])],
                        mode='lines',
                        line=dict(color=color, width=2.5, dash='dot'),
                        opacity=0.6,
                        showlegend=False,
                        hoverinfo='skip'
                    ))
            
            frame_list = trace_list + frame_list
        
        frames.append(go.Frame(data=frame_list, name=str(year)))
    
    fig.frames = frames
    
    sliders = [{
        'active': 0,
        'currentvalue': {'prefix': 'Year: ', 'font': {'size': 16}},
        'pad': {'t': 50},
        'len': 0.9,
        'x': 0.05,
        'xanchor': 'left',
        'y': -0.25,
        'yanchor': 'top',
        'steps': [{'args': [[str(year)], {'frame': {'duration': 300, 'redraw': True}, 'mode': 'immediate'}], 'label': str(year), 'method': 'animate'} for year in years]
    }]
    
    trace_status = "ON" if show_trace else "OFF"
    fig.update_layout(
        title=dict(text=f'AI Impact on Industries: Interactive Time Series (Trace: {trace_status})', font=dict(size=20), x=0.5),
        xaxis=dict(title='AI Intensity Score', gridcolor='#e2e8f0', zeroline=False, range=[df['avgAIIntensity'].min() - 0.05, df['avgAIIntensity'].max() + 0.05]),
        yaxis=dict(title='Average Salary (USD)', gridcolor='#e2e8f0', tickformat='$,.0f', zeroline=False),
        template='plotly_white',
        width=950,
        height=750,
        margin=dict(b=280, l=80, r=80, t=100),
        legend=dict(title='Industries (Click to toggle)', orientation='h', yanchor='top', y=-0.35, xanchor='center', x=0.5),
        hoverlabel=dict(bgcolor='white', font_size=12),
        updatemenus=[{
            'type': 'buttons',
            'showactive': False,
            'x': 0.05,
            'y': -0.08,
            'xanchor': 'left',
            'yanchor': 'top',
            'pad': {'r': 10, 't': 10},
            'buttons': [
                {'label': '▶ Play', 'method': 'animate', 'args': [None, {'frame': {'duration': 500, 'redraw': True}, 'fromcurrent': True}]},
                {'label': '⏸ Pause', 'method': 'animate', 'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]},
                {'label': '🔴 Hide Trace' if show_trace else '🟢 Show Trace', 'method': 'restyle', 'args': [{'visible': [not show_trace if hasattr(fig.data[i], 'mode') and fig.data[i].mode == "lines" else True for i in range(len(fig.data))]}]}
            ]
        }],
        sliders=sliders
    )
    
    return fig

print("🎨 Generating interactive bubble chart with slider (Trace ON)...")
fig_bubble_slider = create_interactive_bubble_chart(df_agg, show_trace=True)
fig_bubble_slider.show()

🎨 Generating interactive bubble chart with slider (Trace ON)...


## 3. Parallel Coordinates - Multi-Metric Trends

Shows time series evolution for all 4 key metrics across industries.

### Metrics Displayed:
1. **AI Intensity Score** - How much AI is integrated into the industry
2. **Average Salary (USD)** - Compensation trends over time
3. **Automation Risk Score** - Probability of job automation
4. **Job Count** - Market size and growth

This replaces the React implementation's scrollable parallel coordinates with a compact subplot layout.

In [19]:
def create_parallel_coordinates_chart(df):
    """
    Create 4-subplot parallel coordinates showing metric trends over time.
    
    Each subplot shows one metric's evolution across all industries.
    Click legend items to toggle industry visibility.
    """
    
    # Create subplots
    fig = make_subplots(
        rows=4, cols=1,
        subplot_titles=(
            '📊 AI Intensity Score',
            '💰 Average Salary (USD)',
            '⚠️ Automation Risk Score',
            '👥 Job Count'
        ),
        vertical_spacing=0.08,
        shared_xaxes=True
    )
    
    industries = sorted(df['industry'].unique())
    years = sorted(df['year'].unique())
    
    metrics = [
        ('avgAIIntensity', ':.2f', ''),
        ('avgSalary', '$,.0f', ''),
        ('avgAutomationRisk', '.0%', ''),
        ('jobCount', ',', '')
    ]
    
    for row_idx, (metric, fmt, suffix) in enumerate(metrics, 1):
        for industry in industries:
            industry_data = df[df['industry'] == industry].sort_values('year')
            color = INDUSTRY_COLORS.get(industry, '#64748b')
            
            fig.add_trace(
                go.Scatter(
                    x=industry_data['year'],
                    y=industry_data[metric],
                    mode='lines+markers',
                    name=industry,
                    line=dict(color=color, width=2.5),
                    marker=dict(size=6, color=color, line=dict(width=1, color='white')),
                    legendgroup=industry,
                    showlegend=(row_idx == 1)  # Show legend only in first subplot
                ),
                row=row_idx,
                col=1
            )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text='Multi-Metric Time Series by Industry (2010-2024)',
            font=dict(size=18),
            x=0.5
        ),
        template='plotly_white',
        width=950,
        height=900,
        showlegend=True,
        legend=dict(
            title='Industries (Click to toggle)',
            orientation='h',
            yanchor='bottom',
            y=-0.05,
            xanchor='center',
            x=0.5
        ),
        hoverlabel=dict(bgcolor='white', font_size=11)
    )
    
    # Update axes for all subplots
    for i in range(1, 5):
        fig.update_yaxes(
            gridcolor='#e2e8f0',
            zeroline=False,
            row=i, col=1
        )
    
    fig.update_xaxes(
        gridcolor='#e2e8f0',
        tickformat='d',
        tickvals=years,
        row=4, col=1
    )
    
    return fig

# Create and display parallel coordinates
print("📈 Generating parallel coordinates chart...")
fig_parallel = create_parallel_coordinates_chart(df_agg)
fig_parallel.show()

📈 Generating parallel coordinates chart...


## 4. Industry Evolution Trace Visualization

This chart shows **complete evolution paths** for all industries across the entire time span.

### Key Differences from React Version:
- ❌ **No real-time rendering**: Unlike React's dynamic trace that updates during drag, this is a static view of the full timeline
- ✅ **Complete paths**: Shows every year's data point connected by lines
- ✅ **Ghost markers**: Dashed circles at each historical position indicate job count changes

### Use Case:
Use this to understand long-term industry trajectories without needing to manually scrub through years.

In [20]:
def create_evolution_trace_chart(df, start_year=None, end_year=None):
    """
    Create simplified evolution trace showing complete industry paths.
    
    This is an INDEPENDENT visualization - not tied to slider interaction.
    Shows how industries move through AI-Salary space over the selected period.
    
    SIMPLIFIED: Removed ghost markers to avoid Plotly compatibility issues.
    """
    
    if start_year is None:
        start_year = df['year'].min()
    if end_year is None:
        end_year = df['year'].max()
    
    fig = go.Figure()
    
    # Filter data for trace period
    trace_data = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    industries = sorted(trace_data['industry'].unique())
    
    for industry in industries:
        industry_path = trace_data[trace_data['industry'] == industry].sort_values('year')
        color = INDUSTRY_COLORS.get(industry, '#64748b')
        
        # Add evolution path (line + markers)
        # Markers size is proportional to job count for this industry
        marker_sizes = [float(np.sqrt(count)) * 2.5 for count in industry_path['jobCount']]
        
        fig.add_trace(go.Scatter(
            x=[float(x) for x in industry_path['avgAIIntensity']],
            y=[float(y) for y in industry_path['avgSalary']],
            mode='lines+markers',
            name=industry,
            line=dict(color=color, width=2.5),
            marker=dict(
                size=marker_sizes,
                color=color,
                line=dict(width=2, color='white')
            ),
            text=[f"{industry}<br>Year: {year}<br>AI: {float(ai):.2f}<br>Salary: ${float(sal):,.0f}<br>Jobs: {int(jobs):,}" 
                  for year, ai, sal, jobs in zip(industry_path['year'], 
                                                  industry_path['avgAIIntensity'],
                                                  industry_path['avgSalary'],
                                                  industry_path['jobCount'])],
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title=dict(
            text=f'Industry Evolution Paths ({start_year}-{end_year})',
            font=dict(size=18),
            x=0.5
        ),
        xaxis=dict(
            title='AI Intensity Score →',
            gridcolor='#e2e8f0',
            zeroline=False
        ),
        yaxis=dict(
            title='Average Salary (USD) →',
            gridcolor='#e2e8f0',
            tickformat='$,.0f',
            zeroline=False
        ),
        template='plotly_white',
        width=950,
        height=650,
        showlegend=True,
        legend=dict(
            title='Industries',
            orientation='h',
            yanchor='bottom',
            y=-0.15,
            xanchor='center',
            x=0.5
        ),
        hoverlabel=dict(bgcolor='white', font_size=12)
    )
    
    # Add annotation explaining the visualization
    fig.add_annotation(
        text="💡 Lines = Position trajectory | Marker size = Job count",
        xref="paper", yref="paper",
        x=0.5, y=-0.22,
        showarrow=False,
        font=dict(size=11, color="#64748b")
    )
    
    return fig

# Create and display trace visualization
print("🔍 Generating evolution trace chart...")
fig_trace = create_evolution_trace_chart(df_agg)
fig_trace.show()

🔍 Generating evolution trace chart...


## 5. Dynamic Leaderboard Table

Replicates the React sidebar ranking functionality with pandas styling.

### Features:
- Sort by any metric (Salary, AI Intensity, Risk, Job Count)
- Toggle ascending/descending order
- Color-coded industry indicators
- Formatted values matching React display

In [21]:
def create_leaderboard_table(df, metric='avgSalary', year=None, ascending=False):
    """
    Create styled leaderboard table showing industry rankings.
    
    Args:
        df: Aggregated dataframe
        metric: Metric to rank by ('avgSalary', 'avgAIIntensity', 'avgAutomationRisk', 'jobCount')
        year: Specific year to display (default: latest year)
        ascending: Sort order (False = descending/highest first)
    """
    
    if year is None:
        year = df['year'].max()
    
    # Filter by year
    year_data = df[df['year'] == year].copy()
    
    # Add color column
    year_data['color'] = year_data['industry'].map(lambda x: INDUSTRY_COLORS.get(x, '#64748b'))
    
    # Sort by metric
    year_data = year_data.sort_values(metric, ascending=ascending)
    year_data['rank'] = range(1, len(year_data) + 1)
    
    # Format values based on metric type
    if metric == 'avgSalary':
        year_data['formatted'] = year_data[metric].apply(lambda x: f'${x/1000:.1f}k')
        metric_label = 'Avg Salary'
    elif metric == 'avgAutomationRisk':
        year_data['formatted'] = year_data[metric].apply(lambda x: f'{x*100:.1f}%')
        metric_label = 'Auto Risk'
    elif metric == 'avgAIIntensity':
        year_data['formatted'] = year_data[metric].apply(lambda x: f'{x:.2f}')
        metric_label = 'AI Intensity'
    else:  # jobCount
        year_data['formatted'] = year_data[metric].apply(lambda x: f'{x:,}')
        metric_label = 'Job Count'
    
    # Create display dataframe
    display_df = year_data[['rank', 'industry', 'formatted', 'color']].copy()
    display_df.columns = ['Rank', 'Industry', metric_label, 'Color']
    
    # Apply styling - Fixed: use col_name from row.index
    styled = display_df.style.apply(
        lambda row: [
            f'background-color: {row["Color"]}22; color: {row["Color"]}; font-weight: bold' if col_name == 'Industry' else ''
            for col_name in row.index
        ],
        axis=1
    )
    
    return styled

# Display multiple leaderboard views
print("\n" + "="*60)
print("📊 LEADERBOARD TABLES")
print("="*60)

latest_year = df_agg['year'].max()
print(f"\n🏆 Top Industries by Average Salary ({latest_year}):")
display(create_leaderboard_table(df_agg, metric='avgSalary', year=latest_year, ascending=False))

print(f"\n🤖 Highest AI Intensity ({latest_year}):")
display(create_leaderboard_table(df_agg, metric='avgAIIntensity', year=latest_year, ascending=False))

print(f"\n⚠️ Highest Automation Risk ({latest_year}):")
display(create_leaderboard_table(df_agg, metric='avgAutomationRisk', year=latest_year, ascending=False))


📊 LEADERBOARD TABLES

🏆 Top Industries by Average Salary (2025):


,Rank,Industry,Avg Salary,Color
138,1,Finance,$91.9k,#d97706
142,2,Retail,$90.3k,#ea580c
137,3,Energy,$84.4k,#0d9488
139,4,Government,$83.6k,#f59e0b
141,5,Manufacturing,$83.3k,#7c3aed
140,6,Healthcare,$82.6k,#db2777
135,7,Agriculture,$82.1k,#16a34a
143,8,Tech,$79.9k,#dc2626
136,9,Education,$78.2k,#0891b2



🤖 Highest AI Intensity (2025):


,Rank,Industry,AI Intensity,Color
137,1,Energy,0.57,#0d9488
138,2,Finance,0.56,#d97706
143,3,Tech,0.54,#dc2626
136,4,Education,0.53,#0891b2
139,5,Government,0.51,#f59e0b
142,6,Retail,0.48,#ea580c
135,7,Agriculture,0.46,#16a34a
140,8,Healthcare,0.39,#db2777
141,9,Manufacturing,0.37,#7c3aed



⚠️ Highest Automation Risk (2025):


,Rank,Industry,Auto Risk,Color
135,1,Agriculture,47.1%,#16a34a
141,2,Manufacturing,46.9%,#7c3aed
140,3,Healthcare,46.1%,#db2777
139,4,Government,43.1%,#f59e0b
142,5,Retail,42.3%,#ea580c
137,6,Energy,39.8%,#0d9488
136,7,Education,36.9%,#0891b2
143,8,Tech,34.3%,#dc2626
138,9,Finance,33.2%,#d97706


## 6. Summary & Implementation Notes

### ✅ What Python/Plotly Achieves:

1. **Data Processing Excellence**
   - `pandas` provides superior aggregation capabilities compared to D3
   - Easy groupby operations replace complex `d3.rollups`

2. **Native Interactivity**
   - Hover tooltips work out-of-the-box
   - Zoom, pan, and box select built-in
   - Legend click-to-toggle requires zero custom code

3. **Slider Widget**
   - Replaces complex D3 drag handlers with simple `frames` + `sliders`
   - Smooth transitions between years
   - Play/Pause buttons for automatic animation

4. **Export Flexibility**
   - Save as standalone HTML files
   - Export to PNG/PDF for reports
   - Embed in web apps or Jupyter notebooks

### ⚠️ Limitations vs React/D3:

1. **Custom Animations**
   - Plotly's transitions are less customizable than D3's `.transition()`
   - Cannot easily implement complex easing functions

2. **State Management**
   - React's component state allows complex interactions (e.g., multi-select, compare modes)
   - Plotly widgets are more limited in stateful interactions

3. **UI Customization**
   - CSS styling in React offers pixel-perfect control
   - Plotly's layout options are more constrained

4. **Complex Overlays**
   - Tutorial spotlight system difficult to replicate in Jupyter
   - Custom tooltip positioning less flexible

### 🔄 State Update Logic Comparison:

**React/D3 Approach:**
```javascript
// Complex event-driven state management
const dragLeft = d3.drag()
  .on('drag', (event) => {
    setCompareYears(prev => {
      const newYear = calculateYearFromPosition(event.x);
      return { ...prev, left: newYear };
    });
  });
```

**Python/Plotly Approach:**
```python
# Pre-computed frames with slider widget
frames = [create_frame_for_year(year) for year in years]
slider = dict(steps=[{'args': [[str(year)]], 'method': 'animate'} for year in years])
# No manual event handling needed!
```

**Key Insight:** Plotly's declarative approach pre-computes all states, while React's imperative approach calculates states on-demand during user interaction.

### 💡 Recommendation:

| Use Case | Recommended Tool |
|----------|------------------|
| Data exploration & analysis | **Python/Plotly** |
| Static reports & presentations | **Python/Plotly** |
| Quick prototyping | **Python/Plotly** |
| Production dashboards | **React/D3** |
| Complex animations | **React/D3** |
| Custom UI/UX requirements | **React/D3** |

---
## Bonus: Export Charts to Standalone HTML

Generate self-contained HTML files that can be shared or embedded in websites.

These files include all necessary JavaScript and data - no server required!

In [22]:
# Export all charts to HTML
output_files = {
    'task2_interactive_bubble.html': fig_bubble_slider,
    'task2_parallel_coords.html': fig_parallel,
    'task2_evolution_trace.html': fig_trace
}

for filename, fig in output_files.items():
    fig.write_html(filename, include_plotlyjs='cdn', full_html=True)
    print(f"✅ Exported: {filename}")

print("\n📁 Generated Files:")
for filename in output_files.keys():
    print(f"   • {filename}")

print("\n💡 Tip: Open these HTML files in any modern browser to interact with the charts!")

✅ Exported: task2_interactive_bubble.html
✅ Exported: task2_parallel_coords.html
✅ Exported: task2_evolution_trace.html

📁 Generated Files:
   • task2_interactive_bubble.html
   • task2_parallel_coords.html
   • task2_evolution_trace.html

💡 Tip: Open these HTML files in any modern browser to interact with the charts!
